<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 2C: *Spatial Join Fire Data*
##### Version Number: 4.0
---
### Contents  
> *Fire Damage Spatial Join of Nearest Sampling Locations*\
> *Fire Ignition Spatial Join of Nearest Sampling Locations with Fire Damage Datasets*\
> *Fire Spread Spatial Join of Nearest Sampling Locations with Fire Damage Datasets*\
> *Export File*
---
### Notes
- Integrate wildfire impact data with daily weather data from the sampling grid.
### Inputs
- Main Dataset - `samples_res.csv` 
- Wildfire Damage Data - `fires_damage.csv` (Appendix B)
- Wildfire Ignition Data - `fires_ignition.csv` (Appendix B)
- Wildfire Spread Data - `fires_spread.csv` (Appendix B)
---
### Outputs  
- `samples_fires.csv` Sampling grid dataset merged with fire damage, spread, and ignition data.
---
### User Created Dependencies  

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Data handling
import pandas as pd
import numpy as np
import geopandas as gpd
import datetime as dt

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import Point, Polygon

---

## Data Loading

In [3]:
samples = pd.read_csv("../data/processed/samples_res.csv")
fire_damages = pd.read_csv("../data/processed/fires_damage.csv")
fire_spread = pd.read_csv('../data/processed/fires_spread.csv')
fire_ignition = pd.read_csv('../data/processed/fires_ignition.csv')

In [4]:
samples['Date'] = pd.to_datetime(samples['Date'])
fire_ignition['Date'] = pd.to_datetime(fire_ignition['Date'])
fire_damages['Date'] = pd.to_datetime(fire_damages['Date'])
fire_spread['Date'] = pd.to_datetime(fire_spread['Date'])

In [5]:
samples = samples.sort_values(
    by=["grid_id", "Date"]
)

#### Subset data for faster loops

In [6]:
keep = ['centroid_easting','centroid_northing','Date','grid_id']
join_samples = samples[keep].copy()

#### Set geometries
- Sample grid ArcGIS work was performed in EPSG 3310 to preserve area. 

In [7]:
join_samples['geometry'] = [Point(xy) for xy in zip(join_samples['centroid_easting'], join_samples['centroid_northing'])]
samples_gdf = gpd.GeoDataFrame(join_samples, geometry='geometry', crs="EPSG:3310")

In [8]:
fire_damages['geometry'] = [Point(xy) for xy in zip(fire_damages['Fire_Longitude'], fire_damages['Fire_Latitude'])]
fire_damages_gdf = gpd.GeoDataFrame(fire_damages, geometry='geometry', crs="EPSG:4326")

## Project to Equal area projection for more accuracy
fire_damages_projected = fire_damages_gdf.to_crs(3310)

In [9]:
fire_ignition['geometry'] = [Point(xy) for xy in zip(fire_ignition['Fire_Longitude'], fire_ignition['Fire_Latitude'])]
fire_ignition_gdf = gpd.GeoDataFrame(fire_ignition, geometry='geometry', crs="EPSG:4326")

## Project to Equal area projection for more accuracy
fire_ignition_projected = fire_ignition_gdf.to_crs(3310)

In [10]:
fire_spread['geometry'] = [Point(xy) for xy in zip(fire_spread['Centroid_x'], fire_spread['Centroid_y'])]
fire_spread_gdf = gpd.GeoDataFrame(fire_spread, geometry='geometry', crs="EPSG:3310")

### Define default buffer
- Centroids are loosely spaced 46000 meters apart. Set buffer distance a small bit above this value. This parameter is adjustable.

In [11]:
# Square Buffer
buffer_size = 23000

### Main loop of Spatial Join algorithms

All three widlfire datasets use the same algorithm.
- Chunk to process by year
- Loop through all dates in samples dataset
- Save all fires and grid centroids associated with the current date
    - If no fires, move to next date
- Create a buffer around each around each fire for current date
- Intersection spatial join of sampling centroids and buffered fires
- Aggregate fields in case of multiple fires in a buffered range
- Assign samples to back to main dataframe

## Fire Damage Spatial Join

Outputs:
- `damage_per_day` - ratio of damage in dollars / days burned
- `total_fire_damage` - sum of all damages done with a sampling grid
- `damage_so_far` - damages caused so far in ongoing fires in each grid

In [12]:
def add_fire_damage_features_for_year(
    samples_gdf,
    fire_damages_projected,
    year,
    buffer_size,
    square_buffer_fn,
):

    # Limit to year chunk
    samples_year = samples_gdf[samples_gdf["Date"].dt.year == year]
    fires_year = fire_damages_projected[
        fire_damages_projected["Date"].dt.year == year
    ]

    # If no samples or fires skip processing
    if samples_year.empty or fires_year.empty:
        return samples_gdf

    for dt in samples_year["Date"].unique():

        # Subset all fires on this date
        fires_today = fires_year[fires_year["Date"] == dt]
        if fires_today.empty:
            continue

        # Subset all grid centroids on this date
        samples_today = samples_year[samples_year["Date"] == dt]
        if samples_today.empty:
            continue

        # Apply buffer to fires
        fires_buffered = fires_today.copy()
        fires_buffered["geometry"] = fires_buffered.geometry.apply(
            lambda p: square_buffer_fn(p, buffer_size)
        )

        # Spatial join of fires to sampling grid
        joined = gpd.sjoin(
            samples_today,
            fires_buffered,
            how="left",
            predicate="intersects"
        )

        # Aggregate per grid cell
        grouped = (
            joined
            .groupby("grid_id")
            .agg(
                total_fire_damage=("Estimated Damage", "sum"),
                Damage_per_day=("Damage per Day", "max"),
            )
            .fillna(0)
        )

        # Write results back to main dataframe
        mask = samples_gdf["Date"] == dt

        samples_gdf.loc[mask, "total_fire_damage"] = (
            samples_gdf.loc[mask, "grid_id"]
            .map(grouped["total_fire_damage"])
            .fillna(0)
        )

    return samples_gdf


In [13]:
# Snapshot for post merge checking
premerged = samples_gdf.copy()

for yr in sorted(samples_gdf["Date"].dt.year.unique()):
    samples_gdf = add_fire_damage_features_for_year(
        samples_gdf=samples_gdf,
        fire_damages_projected=fire_damages_projected,
        year=yr,
        buffer_size=buffer_size,
        square_buffer_fn=square_buffer,
    )

In [14]:
post_merge_check(samples_gdf, premerged)

Premerged shape:  (608880, 5)
Merged shape:  (608880, 6)


Duplicates before merge:  0


Duplicates after merge:  0
NA values before merge:  0
NA values after merge:  565928


In [15]:
samples_gdf.isna().sum()

centroid_easting          0
centroid_northing         0
Date                      0
grid_id                   0
geometry                  0
total_fire_damage    565928
dtype: int64

**Note**: NA values are for buffered areas without any fires in range within the date range. Fill these values with 0. 

In [16]:
samples_gdf = samples_gdf.fillna(0)

## Fire Ignition Spatial Join

Outputs:
- `fire_count` - count of all fires in a sampling grid

In [17]:
def add_fire_ignition_features_for_year(
    samples_gdf,
    fire_ignition_projected,
    year,
    buffer_size,
    square_buffer_fn,
):


    # Limit to year chunk
    samples_year = samples_gdf[samples_gdf["Date"].dt.year == year]
    fires_year = fire_ignition_projected[
        fire_ignition_projected["Date"].dt.year == year
    ]

    # If no samples or fires, skip year
    if samples_year.empty or fires_year.empty:
        return samples_gdf

    for dt in samples_year["Date"].unique():

        # Subset all fires on this date
        fires_today = fires_year[fires_year["Date"] == dt]
        if fires_today.empty:
            continue

        # Subset all grid centroids on this date
        samples_today = samples_year[samples_year["Date"] == dt]
        if samples_today.empty:
            continue

        # Buffer ignitions
        fires_buffered = fires_today.copy()
        fires_buffered["geometry"] = fires_buffered.geometry.apply(
            lambda p: square_buffer_fn(p, buffer_size)
        )

        # Spatial join fires to sampling grid
        joined = gpd.sjoin(
            samples_today,
            fires_buffered,
            how="left",
            predicate="intersects"
        )

        # Aggregate per grid cell
        grouped = (
            joined
            .groupby("grid_id")
            .agg(
                total_fire_count=("Fire Name", "count")
            )
            .fillna(0)
        )

        # Write results back to main dataframe
        mask = samples_gdf["Date"] == dt
        samples_gdf.loc[mask, "fire_count"] = (
            samples_gdf.loc[mask, "grid_id"]
            .map(grouped["total_fire_count"])
            .fillna(0)
        )

    return samples_gdf


In [18]:
# Snapshot for post merge checks
premerged = samples_gdf.copy()

for yr in sorted(samples_gdf["Date"].dt.year.unique()):
    samples_gdf = add_fire_ignition_features_for_year(
        samples_gdf=samples_gdf,
        fire_ignition_projected=fire_ignition_projected,
        year=yr,
        buffer_size=buffer_size,
        square_buffer_fn=square_buffer,
    )


In [19]:
post_merge_check(samples_gdf, premerged)

Premerged shape:  (608880, 6)
Merged shape:  (608880, 7)


Duplicates before merge:  0


Duplicates after merge:  0
NA values before merge:  0
NA values after merge:  6136


In [20]:
samples_gdf.isna().sum()

centroid_easting        0
centroid_northing       0
Date                    0
grid_id                 0
geometry                0
total_fire_damage       0
fire_count           6136
dtype: int64

**Note**: NA values caused by buffered areas without any fires in range within the date range. Fill these values with 0. 

In [21]:
samples_gdf = samples_gdf.fillna(0)

## Fire Spread Spatial Join

Outputs:
- `acres` - sum of all acres burned in a sampling grid
- `acres_per_day` - amount of acres burnt / days fire burnt
- `acres_burned_so_far` - running total of acres burnt while a wildfire is burning

In [22]:
def add_fire_spread_features_for_year(
    samples_gdf,
    fire_spread_gdf,
    year,
    buffer_size,
    square_buffer_fn,
):

    # Limit to year chunk
    samples_year = samples_gdf[samples_gdf["Date"].dt.year == year]
    fires_year = fire_spread_gdf[
        fire_spread_gdf["Date"].dt.year == year
    ]
    
    # if no samples or fires skip year
    if samples_year.empty or fires_year.empty:
        return samples_gdf

    for dt in samples_year["Date"].unique():

        # Subset all fires in currrent date
        fires_today = fires_year[fires_year["Date"] == dt]
        if fires_today.empty:
            continue

        # Subset all sampling grid centroids for current date
        samples_today = samples_year[samples_year["Date"] == dt]
        if samples_today.empty:
            continue

        # Buffer fire spread geometries
        fires_buffered = fires_today.copy()
        fires_buffered["geometry"] = fires_buffered.geometry.apply(
            lambda p: square_buffer_fn(p, buffer_size)
        )

        # Spatial join fires to sampling grid
        joined = gpd.sjoin(
            samples_today,
            fires_buffered,
            how="left",
            predicate="intersects"
        )

        # Aggregate per grid cell
        grouped = (
            joined
            .groupby("grid_id")
            .agg(
                Acres=("Acres", "sum"),
            )
            .fillna(0)
        )

        # Write results back to main dataframe
        mask = samples_gdf["Date"] == dt

        samples_gdf.loc[mask, "acres"] = (
            samples_gdf.loc[mask, "grid_id"]
            .map(grouped["Acres"])
            .fillna(0)
        )

    return samples_gdf


In [23]:
# Snapshot for post merge checks
premerged = samples_gdf.copy()

for yr in sorted(samples_gdf["Date"].dt.year.unique()):
    samples_gdf = add_fire_spread_features_for_year(
        samples_gdf=samples_gdf,
        fire_spread_gdf=fire_spread_gdf,
        year=yr,
        buffer_size=buffer_size,
        square_buffer_fn=square_buffer,
    )


In [24]:
post_merge_check(samples_gdf, premerged)

Premerged shape:  (608880, 7)
Merged shape:  (608880, 8)


Duplicates before merge:  0


Duplicates after merge:  0
NA values before merge:  0
NA values after merge:  329928


In [25]:
samples_gdf.isna().sum()

centroid_easting          0
centroid_northing         0
Date                      0
grid_id                   0
geometry                  0
total_fire_damage         0
fire_count                0
acres                329928
dtype: int64

**Note**: NA values caused by buffered areas without any widespread fires in range within the date range. Fill these values with 0. 

In [26]:
samples_gdf = samples_gdf.fillna(0)

## Merge back to main dataset

In [27]:
samples_gdf = samples_gdf.drop(columns=['centroid_easting','centroid_northing','geometry'])

In [28]:
# Merge on BOTH station and date
samples_fires = samples.merge(
    samples_gdf,
    on=['grid_id','Date'],
    how='left'
)

In [29]:
post_merge_check(samples_fires, samples)

Premerged shape:  (608880, 68)
Merged shape:  (608880, 71)


Duplicates before merge:  0


Duplicates after merge:  0
NA values before merge:  0


NA values after merge:  0


## Export File

In [30]:
samples_fires.to_csv('../data/processed/samples_fires.csv',index=False)
print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
